In [2]:
import numpy as np
from scipy.stats import norm
import pandas as pd


In [9]:
def power_analysis_calculator(alpha=0.05, power=0.80, target_d=0.5, verbose=True):
    """
    Calculate sample sizes for AI intervention study with two analyses.
    
    Parameters:
    -----------
    alpha : float
        Significance level (default: 0.05)
    power : float
        Desired power (default: 0.80)
    target_d : float
        Target effect size (Cohen's d) (default: 0.5)
    verbose : bool
        Print detailed output (default: True)
    
    Returns:
    --------
    dict : Results dictionary with sample sizes and detectable effect sizes
    """
    
    # Critical values
    z_alpha = norm.ppf(1 - alpha/2)  # Two-tailed test
    z_beta = norm.ppf(power)
    
    # Base calculation for balanced design
    base_n_per_group = 2 * ((z_alpha + z_beta) / target_d) ** 2
    base_n = base_n_per_group * 2 
     
    if verbose:
        print(f"Power Analysis Parameters:")
        print(f"Alpha = {alpha}, Power = {power}, Target d = {target_d}")
        print(f"Z_alpha = {z_alpha:.3f}, Z_beta = {z_beta:.3f}")
        print(f"Base n per group = {base_n_per_group:.1f}")
        print()
    
    # Analysis A: Attribute effects (4n vs n, unbalanced)
    # High condition: 4 groups × 5 scenarios = 20n total
    # Low condition: 1 group × 5 scenarios = 5n total  
    # Effective n = (2 × 20n × 5n) / (20n + 5n) = 8n
    eff_mult_a = 8  # Effective multiplier for Analysis A
    n_per_cell_a = int(np.ceil(base_n_per_group / eff_mult_a))
    total_a = n_per_cell_a * 25  # 25 groups in Analysis A
    
    # Analysis B: Transparency effects (n vs n, balanced)
    # Each condition: n × 5 scenarios = 5n
    # Total effective n = 2 × 5n = 10n
    # Need: 5n = base_n_per_group, so n = base_n_per_group/5
    n_per_cell_b = int(np.ceil(base_n_per_group / 5))
    additional_b = n_per_cell_b * 5  # 5 new groups for Analysis B
    
    # Use larger requirement
    n_recommended = max(n_per_cell_a, n_per_cell_b)
    total_participants = n_recommended * 30  # 30 total unique groups
    
    # Calculate detectable effect sizes with recommended n
    detectable_d_a = (z_alpha + z_beta) / np.sqrt(eff_mult_a * n_recommended / 2)
    detectable_d_b = (z_alpha + z_beta) / np.sqrt(5 * n_recommended)
    
    if verbose:
        print(f"Analysis A (Attribute Effects - 20n vs 5n):")
        print(f"  Required: {n_per_cell_a} per cell")
        print(f"  Total: {total_a} participants")
        print()
        print(f"Analysis B (Transparency Effects - 5n vs 5n):")
        print(f"  Required: {n_per_cell_b} per cell")  
        print(f"  Additional: {additional_b} participants")
        print()
        print(f"RECOMMENDATION: {n_recommended} per cell ({total_participants} total)")
        print(f"  Attribute effects: can detect d >= {detectable_d_a:.2f}")
        print(f"  Transparency effects: can detect d >= {detectable_d_b:.2f}")
        print()
    
    return {
        'n_per_cell_recommended': n_recommended,
        'total_participants': total_participants,
        'n_analysis_a': n_per_cell_a,
        'n_analysis_b': n_per_cell_b,
        'detectable_d_attributes': detectable_d_a,
        'detectable_d_transparency': detectable_d_b,
        'base_n': base_n
    }

def calculate_detectable_d(n_per_cell, alpha=0.05, power=0.80):
    """
    Calculate what effect size can be detected with given sample size.
    
    Parameters:
    -----------
    n_per_cell : int
        Sample size per cell
    alpha : float
        Significance level
    power : float
        Power level
    
    Returns:
    --------
    dict : Detectable effect sizes for both analyses
    """
    
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    
    # Analysis A: 4n vs n (effective multiplier = 8)
    d_attributes = (z_alpha + z_beta) / np.sqrt(8 * n_per_cell / 2)
    
    # Analysis B: n×5 vs n×5 (effective n per condition = 5n)
    d_transparency = (z_alpha + z_beta) / np.sqrt(5 * n_per_cell)
    
    return {
        'detectable_d_attributes': d_attributes,
        'detectable_d_transparency': d_transparency
    }

# Standard analysis
print("1. STANDARD ANALYSIS (5 scenarios):")
print("-" * 40)
results = power_analysis_calculator(alpha=0.05, power=0.80, target_d=0.5)
print()

print("Conservative (detect large effects only):")
conservative = power_analysis_calculator(alpha=0.05, power=0.80, target_d=0.8, verbose=False)
print(f"  {conservative['n_per_cell_recommended']} per cell ({conservative['total_participants']} total)")
print()

print("High sensitivity (detect small effects):")
sensitive = power_analysis_calculator(alpha=0.05, power=0.90, target_d=0.3, verbose=False)
print(f"  {sensitive['n_per_cell_recommended']} per cell ({sensitive['total_participants']} total)")
print()


1. STANDARD ANALYSIS (5 scenarios):
----------------------------------------
Power Analysis Parameters:
Alpha = 0.05, Power = 0.8, Target d = 0.5
Z_alpha = 1.960, Z_beta = 0.842
Base n per group = 62.8

Analysis A (Attribute Effects - 20n vs 5n):
  Required: 8 per cell
  Total: 200 participants

Analysis B (Transparency Effects - 5n vs 5n):
  Required: 13 per cell
  Additional: 65 participants

RECOMMENDATION: 13 per cell (390 total)
  Attribute effects: can detect d >= 0.39
  Transparency effects: can detect d >= 0.35


Conservative (detect large effects only):
  5 per cell (150 total)

High sensitivity (detect small effects):
  47 per cell (1410 total)

